<a href="https://colab.research.google.com/github/RobJavVar/DataSciencePsychNeuro/blob/master/ExerciseSubmissions/10_mixed-effects-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 10: Mixed effects

This homework assignment is designed to give you practice fitting and interpreting mixed effects models.

We will be using the **LexicalData.csv** and **Items.csv** files from the *Homework/lexDat* folder in the class GitHub repository again.

This data is a subset of the [English Lexicon Project database](https://elexicon.wustl.edu/). It provides the reaction times (in milliseconds) of many subjects as they are presented with letter strings and asked to decide, as quickly and as accurately as possible, whether the letter string is a word or not. The **Items.csv** provides characteristics of the words used, namely frequency (how common is this word?) and length (how many letters?). Unlike in the previous homework, there isn't any missing data in the **LexicalData.csv** file.

*Data courtesy of Balota, D.A., Yap, M.J., Cortese, M.J., Hutchison, K.A., Kessler, B., Loftis, B., Neely, J.H., Nelson, D.L., Simpson, G.B., & Treiman, R. (2007). The English Lexicon Project. Behavior Research Methods, 39, 445-459.*

---
## 1. Loading and formatting the data (1 point)

Load in data from the **LexicalData.csv** and **Items.csv** files. As in the previous homeworks, remove the commas from the reaction times and convert them from strings to numbers. Use `left_join` to add word characteristics `Length` and `Log_Freq_Hal` from **Items** to **LexicalData**.

*Note: the `Freq_HAL` variable in **Items.csv** has a similar formatting issue, using string values with commas. We're not going to worry about fixing this since we're only using `Log_Freq_HAL`, which is the natural log transformation of `Freq_HAL`, in this homework.*

In [2]:
library(tidyverse)
lexical <- read_csv("LexicalData.csv")
items <- read_csv("Items.csv")

lexical <- lexical %>%
  mutate(D_RT = as.numeric(gsub(",", "", D_RT)))

── Attaching core tidyverse packages ──────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Rows: 62610 Columns: 7
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: ","
chr (1): D_Word
dbl (4): Sub_ID, Trial, Type, D_Zscore
num (1): D_RT
lgl (1): Outlier

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this messa

In [3]:
lexical_data <- lexical %>%
  left_join(items %>% select(Word, Length, Log_Freq_HAL), by = c("D_Word" = "Word")) 
head(lexical_data)

Sub_ID,Trial,Type,D_RT,D_Word,Outlier,D_Zscore,Length,Log_Freq_HAL
<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<lgl>,<dbl>,<dbl>,<dbl>
157,1,1,710,browse,FALSE,-0.437,6,8.856
67,1,1,1094,refrigerant,FALSE,0.825,11,4.644
120,1,1,587,gaining,FALSE,-0.645,7,8.304
21,1,1,984,cheerless,FALSE,0.025,9,2.639
236,1,1,577,pattered,FALSE,-0.763,8,1.386
236,2,1,715,conjures,FALSE,-0.364,8,5.268


---
## 2. Model fitting (4 points)

First, fit a linear model with `Log_Freq_HAL` and `Length` as predictors, and `D_RT` as the output. Include an interaction term. Use `summary()` to look at the model output.

In [4]:
lm_lexical <- lm(D_RT ~ Length * Log_Freq_HAL, data = lexical_data)
summary(lm_lexical)


Call:
lm(formula = D_RT ~ Length * Log_Freq_HAL, data = lexical_data)

Residuals:
     Min       1Q   Median       3Q      Max 
-1118.01  -205.23   -86.95    90.77  3147.07 

Coefficients:
                    Estimate Std. Error t value Pr(>|t|)    
(Intercept)         610.1903    14.6678  41.601  < 2e-16 ***
Length               47.7531     1.6368  29.175  < 2e-16 ***
Log_Freq_HAL         -6.0239     1.9678  -3.061  0.00221 ** 
Length:Log_Freq_HAL  -2.9421     0.2348 -12.528  < 2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 359.1 on 62606 degrees of freedom
Multiple R-squared:  0.09473,	Adjusted R-squared:  0.09469 
F-statistic:  2184 on 3 and 62606 DF,  p-value: < 2.2e-16


Now, install `lme4` using `install.packages()` and then load the library.

In [5]:
library(lme4)

Loading required package: Matrix


Attaching package: 'Matrix'


The following objects are masked from 'package:tidyr':

    expand, pack, unpack




Now fit a mixed effects model that includes the same predictors as the linear model above, as well as random intercepts for `Sub_ID` (i.e., cases where subject ID shifts the RT mean). Use `summary()` to look at the model output.

In [8]:
mixed_lexical <- lmer(D_RT ~ Length * Log_Freq_HAL + (1 | Sub_ID), data = lexical_data)
summary(mixed_lexical)

Linear mixed model fit by REML ['lmerMod']
Formula: D_RT ~ Length * Log_Freq_HAL + (1 | Sub_ID)
   Data: lexical_data

REML criterion at convergence: 888235.6

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-4.5058 -0.5472 -0.1568  0.3103 10.7381 

Random effects:
 Groups   Name        Variance Std.Dev.
 Sub_ID   (Intercept) 46333    215.3   
 Residual             82978    288.1   
Number of obs: 62610, groups:  Sub_ID, 299

Fixed effects:
                    Estimate Std. Error t value
(Intercept)         616.8445    17.1522  35.963
Length               47.7477     1.3162  36.277
Log_Freq_HAL         -7.4374     1.5830  -4.698
Length:Log_Freq_HAL  -2.8778     0.1888 -15.239

Correlation of Fixed Effects:
            (Intr) Length L_F_HA
Length      -0.656              
Log_Frq_HAL -0.645  0.917       
Lng:L_F_HAL  0.582 -0.923 -0.942

---
## 3. Model assessment (4 points)

Compare the three t-values for the fixed effects and the mixed effects models. How do they differ, and why?

> The t-values in the mixed-effects model for the fixed effects were all larger in magnitude for example with length at 36.28 while the linear model was at 29.18. This larger magnitude was due to the inclusion of the random intercept for the mixed effects model of subject ID which then considers the individual differences for reaction time. Then by addressing the variability there is a decrease in residual error in the mixed effect model this indicates that mixed effect model performed better. 

Use the Aikeke Information Criterion (AIC) to compare these two models. Which one is better?

In [9]:
AIC (lm_lexical, mixed_lexical)


,df,AIC
,<dbl>,<dbl>
lm_lexical,5,914436.4
mixed_lexical,6,888247.6


> AIC dictates that mixed effects model was better for the lexical data with the lower AIC score confirming that it was best to address a random effect of subject ID for individual differences 
>

---
##  4. Reflection (1 point)

What other random effects could be controlled for in this data set?

> Another random effect that could be controlled is D_word of the list of different words, as we did with subject ID of doing a 1 over Subject ID we would also do (1|D_Word) to also address of individual va

**DUE:** 5pm EST, March 5, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *n/a*